##Silver to Gold AI


In [0]:
%pip install openai

In [0]:
import openai
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

# 1. Setup
storage_account = "rgstoragecustomer"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/customer_reviews_cleaned"
gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/customer_insights"

# AI Credentials (Ideally, use the Secret Scope we tried earlier!)
AI_KEY = dbutils.secrets.get(scope="project-secrets", key="openai-api-key")
AI_ENDPOINT = "https://suman-mlakaho5-eastus2.cognitiveservices.azure.com/openai/deployments/sentiment-model/chat/completions?api-version=2025-01-01-preview"

# 2. Define the AI Logic
def analyze_review_with_gpt(review_text):
    if not review_text: return "None"
    
    client = openai.AzureOpenAI(
        api_key=AI_KEY, 
        azure_endpoint=AI_ENDPOINT, 
        api_version="2024-02-01"
    )
    
    # We ask for a structured response: Sentiment | Root Cause
    prompt = f"Analyze this customer review: '{review_text}'. Return only the sentiment and one word for the main issue (e.g., 'Negative | Speed')."
    
    try:
        response = client.chat.completions.create(
            model="sentiment-model",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20
        )
        return response.choices[0].message.content
    except:
        return "Error"

# 3. Convert to Spark UDF
ai_udf = udf(analyze_review_with_gpt, StringType())

# 4. Process a Sample (To save credits)
silver_df = spark.read.format("delta").load(silver_path)
sample_df = silver_df.limit(100) # Let's start with 100 rows to test

gold_df = sample_df.withColumn("ai_insight", ai_udf(col("review_text")))

# 5. Save to Gold
gold_df.write.format("delta").mode("overwrite").save(gold_path)

print("Gold Layer with AI Insights Created!")

In [0]:
gold_df.show()